# InstaWorker POC Scope Estimation

**Goal:** Starting from all Salesforce cases, determine how many are within InstaWorker's POC scope — before running the extraction pipeline.

**Approach:** Apply exclusion checks using SF case fields, Databricks customer data, SX order data, and customer type segments. Cases that survive all exclusions are candidates for the pipeline.

**Data files required (place in same directory as this notebook):**
- `cases_ai_order_entry_20260327_002410__1_.xlsx` — Salesforce cases
- `databricks_views_20260403_120547__1_.xlsx` — Databricks customer views
- `Feb_Mar_26_Man_Orders.xlsx` — SX order dump
- `Customer_Types__1_.xlsx` — Customer type codes mapped to segments

## Step 0: Load all data

In [13]:
import pandas as pd
import re
from collections import Counter

# Salesforce cases
sf = pd.read_excel('/Users/mahirsudad/Documents/InstaBrief/cases_ai_order_entry.xlsx', sheet_name='Cases', dtype=str)
sf.columns = sf.columns.str.strip()
print(f"SF cases loaded: {len(sf)}")

# Databricks customer details
db = pd.read_excel('/Users/mahirsudad/Documents/InstaBrief/databricks.xlsx', sheet_name='v_customer_details', dtype=str)
db.columns = db.columns.str.strip()
print(f"Databricks customers loaded: {len(db)}")

# Databricks ship-tos
db_ship = pd.read_excel('/Users/mahirsudad/Documents/InstaBrief/databricks.xlsx', sheet_name='v_customer_ship_tos', dtype=str)
db_ship.columns = db_ship.columns.str.strip()
print(f"Databricks ship-tos loaded: {len(db_ship)}")

# SX order dump
sx = pd.read_excel('/Users/mahirsudad/Documents/InstaBrief/Feb_Mar_26_Man_Orders.xlsx', sheet_name='Instalily_Order_Dump', dtype=str)
sx.columns = sx.columns.str.strip()
print(f"SX line items loaded: {len(sx)}")
print(f"SX unique orders: {sx['OrderNumber'].nunique()}")

# Customer types
ct = pd.read_excel('/Users/mahirsudad/Documents/InstaBrief/Customer_Types.xlsx', dtype=str)
ct.columns = ct.columns.str.strip()
print(f"Customer type codes loaded: {len(ct)}")

SF cases loaded: 2916
Databricks customers loaded: 354603
Databricks ship-tos loaded: 18649
SX line items loaded: 118207
SX unique orders: 60722
Customer type codes loaded: 727


## Step 1: Explore the SF data

Before applying any filters, let's understand what we're working with.

In [14]:
print(f"Total cases: {len(sf)}")
print(f"Date range: {sf['Date/Time Opened'].str[:10].min()} to {sf['Date/Time Opened'].str[:10].max()}")
print(f"\nAccount Number populated: {sf['Account Number'].notna().sum()} ({sf['Account Number'].notna().mean()*100:.1f}%)")
print(f"Account Name populated: {sf['Account Name'].notna().sum()} ({sf['Account Name'].notna().mean()*100:.1f}%)")
print(f"Contact Phone populated: {sf['Contact Phone'].notna().sum()}")
print(f"Contact Email populated: {sf['Contact Email'].notna().sum()}")
print(f"\nOrigin values:")
print(sf['Origin'].value_counts().to_string())
print(f"\nAttachment Count distribution:")
sf['Attachment Count'] = pd.to_numeric(sf['Attachment Count'], errors='coerce').fillna(0)
print(f"  Has attachment: {(sf['Attachment Count'] > 0).sum()}")
print(f"  No attachment: {(sf['Attachment Count'] == 0).sum()}")

Total cases: 2916
Date range: 2026-03-06 to 2026-03-27

Account Number populated: 1344 (46.1%)
Account Name populated: 1350 (46.3%)
Contact Phone populated: 1236
Contact Email populated: 2916

Origin values:
Origin
AIOrderentry@dfsupply.com    2916

Attachment Count distribution:
  Has attachment: 2064
  No attachment: 852


## Step 2: Understand the Email To field

The `Origin` field shows `AIOrderentry@dfsupply.com` for all cases (this is the test queue routing). The actual inbox the customer emailed is in `Email To`. This is our proxy for Case Origin — the field the pipeline checks in Step 3.

In [15]:
# Parse which orders@ inbox each case was sent to
def get_inbox(email_to):
    if pd.isna(email_to):
        return 'BLANK'
    email_to = str(email_to).lower()
    if 'orders@allpointsfps.com' in email_to:
        return 'orders@allpointsfps.com'
    elif 'orders@tundrafmp.com' in email_to:
        return 'orders@tundrafmp.com'
    elif 'orders@dfsupply.com' in email_to:
        return 'orders@dfsupply.com'
    elif 'aiorderentry@dfsupply.com' in email_to:
        return 'aiorderentry@dfsupply.com'
    else:
        return 'other'

sf['inbox'] = sf['Email To'].apply(get_inbox)
print("Cases by inbox:")
print(sf['inbox'].value_counts().to_string())
print(f"\nSample 'other' Email To values:")
other_emails = sf[sf['inbox'] == 'other']['Email To'].str.lower().str.split(';').str[0].value_counts().head(15)
print(other_emails.to_string())

Cases by inbox:
inbox
orders@allpointsfps.com      1583
orders@tundrafmp.com          582
aiorderentry@dfsupply.com     499
orders@dfsupply.com           127
other                         120
BLANK                           5

Sample 'other' Email To values:
Email To
thudley@tundrafmp.com            85
bksales@tundrafmp.com             6
bmurphy@tundrafmp.com             6
bksales@fmponline.com             3
custserv@allpointsfps.com         3
esmith@allpointsfps.com           2
lzvyagintsev@allpointsfps.com     1
jchaikin@allpointsfps.com         1
parts.mtsequipment@gmail.com      1
sdigioia@c-tdesign.com            1
philipjiang@jxtcompany.com        1
gnatal@allpointsfps.com           1
apoppaw@commercialkitchen.com     1
dsorderswest@tundrafmp.com        1
jduda@allpointsfps.com            1


## Step 3: Understand the customer types

We have 728 customer type codes mapped to 15 segments. Based on the April 6 call with Joanne, our includable segments are **Service**, **Service Companies**, and **Institutions**. Everything in **Dealers**, **International**, **Intercompany**, **Wholesaler**, and **Chains** is excluded. Several segments are pending review with Susan.

In [16]:
print("Segments and code counts:")
print(ct['user4'].value_counts().to_string())

# Define exclusions
excluded_segments = {'Dealers', 'International', 'Intercompany', 'Wholesaler', 'Chains'}
excluded_codes = set(ct[ct['user4'].isin(excluded_segments)]['codeval'].astype(str))
print(f"\nExcluded segment codes: {len(excluded_codes)}")
print(f"Excluded segments: {excluded_segments}")

# Pending segments (not excluding yet, not confirmed includable)
includable_segments = {'Service', 'Service Companies', 'Institutions'}
pending_segments = set(ct['user4'].dropna().unique()) - excluded_segments - includable_segments
print(f"\nIncludable segments: {includable_segments}")
print(f"Pending Susan review: {pending_segments}")

Segments and code counts:
user4
Chains               545
Service               42
International         33
Dealers               26
Institutions          23
Other Foodservice     17
Competitors            5
Other                  5
Service Companies      5
Wholesaler             4
Manufacturer           3
Foodservice            1
Design                 1
Intercompany           1

Excluded segment codes: 609
Excluded segments: {'International', 'Wholesaler', 'Dealers', 'Intercompany', 'Chains'}

Includable segments: {'Service', 'Institutions', 'Service Companies'}
Pending Susan review: {'Competitors', 'Design', 'Foodservice', 'Other', 'Other Foodservice', 'Manufacturer'}


## Step 4: Prepare join keys

SF Account Numbers sometimes have a `-1` suffix that needs to be stripped before joining to Databricks `customer_id`.

In [17]:
# Clean SF Account Number — strip -1 suffix
sf['acct_clean'] = sf['Account Number'].apply(
    lambda x: str(x).strip().split('-')[0] if pd.notna(x) and str(x).strip() else None
)

print(f"Account Numbers with -1 suffix: {sf['Account Number'].dropna().str.contains('-1').sum()}")
print(f"Account Numbers without suffix: {sf['Account Number'].dropna().str.contains('-1').eq(False).sum()}")
print(f"\nSample cleaned Account Numbers:")
print(sf[sf['acct_clean'].notna()][['Account Number', 'acct_clean']].drop_duplicates().head(10).to_string(index=False))

Account Numbers with -1 suffix: 710
Account Numbers without suffix: 634

Sample cleaned Account Numbers:
Account Number acct_clean
        544169     544169
       20739-1      20739
        1013-1       1013
        4187-1       4187
      537818-1     537818
         19089      19089
      700172-1     700172
    52012044-1   52012044
         20971      20971
       14535-1      14535


## Step 5: Join SF to Databricks

For cases that have an Account Number, join to Databricks to get `customer_type` and `brand`.

In [18]:
# Prepare Databricks lookup
db_lookup = db[['customer_id', 'customer_type', 'brand']].copy()
db_lookup['customer_id'] = db_lookup['customer_id'].str.strip()

# Join
sf = sf.merge(db_lookup, left_on='acct_clean', right_on='customer_id', how='left', suffixes=('', '_db'))

has_acct = sf['acct_clean'].notna()
joined = sf['customer_id'].notna()

print(f"Cases with Account Number: {has_acct.sum()}")
print(f"  Joined to Databricks: {(has_acct & joined).sum()}")
print(f"  NOT found in Databricks: {(has_acct & ~joined).sum()}")
print(f"Cases without Account Number: {(~has_acct).sum()}")

Cases with Account Number: 1353
  Joined to Databricks: 819
  NOT found in Databricks: 534
Cases without Account Number: 1572


## Step 6: Map customer types to segments

In [19]:
# Map customer_type to segment
ct_map = ct.set_index('codeval')['user4'].to_dict()
ct_desc = ct.set_index('codeval')['descrip'].to_dict()
sf['segment'] = sf['customer_type'].map(ct_map)
sf['type_desc'] = sf['customer_type'].map(ct_desc)

print("Segments for cases that joined to Databricks:")
print(sf[sf['customer_id'].notna()]['segment'].value_counts().to_string())

Segments for cases that joined to Databricks:
segment
Dealers              491
Service              190
Other                 56
Competitors           34
Intercompany          19
Institutions          12
Other Foodservice      2
International          2
Chains                 2
Manufacturer           1


## Step 7: Apply exclusion checks

Now we apply the three checks. A case is **excluded** only if we have enough data to be certain it's out of scope. If we can't determine, it goes to the pipeline.

**Check 1:** `Email To` must contain one of the three orders@ inboxes or aiorderentry@. If it doesn't, exclude.

**Check 2:** If Account Number is populated AND joins to Databricks AND customer type is in an excluded segment → exclude.

**Check 3:** If Account Number is populated AND joins to Databricks AND brand doesn't match the inbox → exclude.

In [20]:
# Start with all cases
sf['excluded'] = False
sf['exclude_reason'] = ''

# CHECK 1: Valid inbox
invalid_inbox = sf['inbox'].isin(['other', 'BLANK'])
sf.loc[invalid_inbox & ~sf['excluded'], 'exclude_reason'] = 'Not sent to valid inbox'
sf.loc[invalid_inbox, 'excluded'] = True

# CHECK 2: Excluded customer type (only where we CAN identify)
known_excluded_type = sf['customer_type'].isin(excluded_codes) & sf['customer_id'].notna()
sf.loc[known_excluded_type & ~sf['excluded'], 'exclude_reason'] = 'Excluded segment: ' + sf.loc[known_excluded_type, 'segment'].fillna('?')
sf.loc[known_excluded_type, 'excluded'] = True

# CHECK 3: Brand mismatch (only where we CAN identify and have a real inbox)
inbox_brand_map = {
    'orders@allpointsfps.com': 'allpoints',
    'orders@tundrafmp.com': 'tundra',
    'orders@dfsupply.com': 'dfs'
}

def check_brand_match(row):
    if pd.isna(row['brand']) or pd.isna(row.get('customer_id')):
        return True  # Can't check, don't exclude
    expected = inbox_brand_map.get(row['inbox'])
    if expected is None:
        return True  # aiorderentry — can't determine brand, don't exclude
    actual = str(row['brand']).lower()
    return expected in actual or actual in expected

brand_ok = sf.apply(check_brand_match, axis=1)
brand_mismatch = ~brand_ok & sf['customer_id'].notna() & ~sf['excluded']
sf.loc[brand_mismatch, 'exclude_reason'] = 'Brand mismatch'
sf.loc[brand_mismatch, 'excluded'] = True

print("EXCLUSION RESULTS")
print("=" * 50)
print(f"Total cases:        {len(sf)}")
print(f"Excluded:           {sf['excluded'].sum()} ({sf['excluded'].mean()*100:.1f}%)")
print(f"Send to pipeline:   {(~sf['excluded']).sum()} ({(~sf['excluded']).mean()*100:.1f}%)")
print(f"\nExclusion reasons:")
print(sf[sf['excluded']]['exclude_reason'].value_counts().to_string())

EXCLUSION RESULTS
Total cases:        2925
Excluded:           660 (22.6%)
Send to pipeline:   2265 (77.4%)

Exclusion reasons:
exclude_reason
Excluded segment: Dealers          485
Not sent to valid inbox            125
Brand mismatch                      34
Excluded segment: Intercompany      12
Excluded segment: International      2
Excluded segment: Chains             2


## Step 8: Try to identify more customers via SX data

For cases with no Account Number, we can try to match them to SX orders using:
1. **Contact Phone** — match SF Contact Phone to SX phone field
2. **PO number from subject line** — parse PO-like strings from the SF Email Subject and match to SX custpo

If we find a match, we get the SX CustomerID, which we can then look up in Databricks to check the customer type.

In [21]:
# Build SX lookup tables
# Phone -> CustomerID
sx_phone_lookup = sx.dropna(subset=['phone']).drop_duplicates(subset=['phone'])
sx_phone_map = dict(zip(sx_phone_lookup['phone'].str.strip(), sx_phone_lookup['CustomerID'].str.replace('.0', '', regex=False)))
print(f"SX phone lookup entries: {len(sx_phone_map)}")

# custpo -> CustomerID (deduplicated to first occurrence)
sx_po_lookup = sx.dropna(subset=['custpo']).drop_duplicates(subset=['custpo'])
sx_po_map = dict(zip(sx_po_lookup['custpo'].str.strip().str.upper(), sx_po_lookup['CustomerID'].str.replace('.0', '', regex=False)))
print(f"SX custpo lookup entries: {len(sx_po_map)}")

SX phone lookup entries: 15842
SX custpo lookup entries: 40759


In [22]:
# For non-excluded cases with no Account Number, try SX matching
pipeline_cases = sf[~sf['excluded']].copy()
no_acct = pipeline_cases[pipeline_cases['acct_clean'].isna()].copy()
print(f"Pipeline cases with no Account Number: {len(no_acct)}")

def try_sx_match(row):
    """Try to find a CustomerID via phone or PO subject match."""
    # Try phone
    phone = str(row['Contact Phone']).strip() if pd.notna(row['Contact Phone']) else ''
    phone_clean = re.sub(r'[^0-9]', '', phone)
    if phone_clean and len(phone_clean) >= 7:
        if phone_clean in sx_phone_map:
            return sx_phone_map[phone_clean], 'phone'
    
    # Try PO from subject
    subject = str(row['Email Subject']).strip() if pd.notna(row['Email Subject']) else ''
    if subject:
        # Extract PO-like patterns
        patterns = re.findall(
            r'(?:Purchase Order|PO|P\.O\.)\s*[#:]?\s*([A-Z0-9\-#]+)',
            subject, re.IGNORECASE
        )
        for po in patterns:
            po_clean = po.strip('#').strip().upper()
            if po_clean in sx_po_map:
                return sx_po_map[po_clean], 'po_subject'
    
    return None, None

no_acct[['sx_customer_id', 'sx_match_method']] = no_acct.apply(
    lambda row: pd.Series(try_sx_match(row)), axis=1
)

matched = no_acct[no_acct['sx_customer_id'].notna()]
print(f"\nMatched to SX: {len(matched)}")
print(f"  By phone: {(matched['sx_match_method'] == 'phone').sum()}")
print(f"  By PO from subject: {(matched['sx_match_method'] == 'po_subject').sum()}")
print(f"  Unmatched: {len(no_acct) - len(matched)}")

Pipeline cases with no Account Number: 1464

Matched to SX: 543
  By phone: 0
  By PO from subject: 543
  Unmatched: 921


In [23]:
# For matched cases, look up customer type in Databricks
db_type_map = dict(zip(db['customer_id'].str.strip(), db['customer_type'].str.strip()))

matched['resolved_type'] = matched['sx_customer_id'].map(db_type_map)
matched['resolved_segment'] = matched['resolved_type'].map(ct_map)

print("Segments for SX-matched cases:")
print(matched['resolved_segment'].value_counts().to_string())

# How many can we now exclude?
newly_excludable = matched[matched['resolved_type'].isin(excluded_codes)]
print(f"\nNewly excludable (excluded segment): {len(newly_excludable)}")
print(newly_excludable['resolved_segment'].value_counts().to_string())

print(f"\nWould remain in pipeline after SX-based exclusion: {len(no_acct) - len(newly_excludable)}")

Segments for SX-matched cases:
resolved_segment
Chains               306
Dealers              151
Service               79
Competitors            5
Other Foodservice      1

Newly excludable (excluded segment): 457
resolved_segment
Chains     306
Dealers    151

Would remain in pipeline after SX-based exclusion: 1007


/var/folders/mc/2x62zj4945qf_99mn1z70bm80000gn/T/ipykernel_19499/2784135817.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched['resolved_type'] = matched['sx_customer_id'].map(db_type_map)
/var/folders/mc/2x62zj4945qf_99mn1z70bm80000gn/T/ipykernel_19499/2784135817.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched['resolved_segment'] = matched['resolved_type'].map(ct_map)


## Step 9: Apply SX-based exclusions and get final count

In [24]:
# Exclude the newly identified out-of-scope cases
newly_excluded_cases = set(newly_excludable['Case Number'])
sf.loc[sf['Case Number'].isin(newly_excluded_cases), 'excluded'] = True
sf.loc[sf['Case Number'].isin(newly_excluded_cases), 'exclude_reason'] = 'SX-matched: excluded segment'

print("FINAL EXCLUSION RESULTS")
print("=" * 60)
print(f"Total cases:           {len(sf)}")
print(f"Total excluded:        {sf['excluded'].sum()} ({sf['excluded'].mean()*100:.1f}%)")
print(f"Send to pipeline:      {(~sf['excluded']).sum()} ({(~sf['excluded']).mean()*100:.1f}%)")
print(f"\nAll exclusion reasons:")
print(sf[sf['excluded']]['exclude_reason'].value_counts().to_string())

FINAL EXCLUSION RESULTS
Total cases:           2925
Total excluded:        1117 (38.2%)
Send to pipeline:      1808 (61.8%)

All exclusion reasons:
exclude_reason
Excluded segment: Dealers          485
SX-matched: excluded segment       457
Not sent to valid inbox            125
Brand mismatch                      34
Excluded segment: Intercompany      12
Excluded segment: International      2
Excluded segment: Chains             2


## Step 10: Profile the pipeline candidates

Let's understand what's going to the pipeline.

In [25]:
pipeline = sf[~sf['excluded']].copy()

print(f"Pipeline candidates: {len(pipeline)}")
print(f"\nBy inbox:")
print(pipeline['inbox'].value_counts().to_string())

print(f"\nAccount Number status:")
has = pipeline['acct_clean'].notna().sum()
no = pipeline['acct_clean'].isna().sum()
print(f"  Has Account Number: {has}")
print(f"  No Account Number (pipeline resolves): {no}")

print(f"\nKnown segments (where identified):")
print(pipeline[pipeline['segment'].notna()]['segment'].value_counts().to_string())

print(f"\nHas attachment:")
print(f"  Yes: {(pipeline['Attachment Count'] > 0).sum()}")
print(f"  No: {(pipeline['Attachment Count'] == 0).sum()}")

print(f"\nCases per day (avg): {len(pipeline) / pipeline['Date/Time Opened'].str[:10].nunique():.1f}")

Pipeline candidates: 1808

By inbox:
inbox
orders@allpointsfps.com      955
aiorderentry@dfsupply.com    499
orders@tundrafmp.com         273
orders@dfsupply.com           81

Account Number status:
  Has Account Number: 801
  No Account Number (pipeline resolves): 1007

Known segments (where identified):
segment
Service              156
Other                 56
Competitors           34
Institutions          10
Other Foodservice      2
Manufacturer           1

Has attachment:
  Yes: 1100
  No: 708

Cases per day (avg): 82.2


## Step 11: Cross-reference pipeline candidates with SX orders

For cases where we can match to an SX order (via CustomerID + custpo), this confirms the case was a real PO that a CSR actually entered. This gives us ground truth for pipeline validation later.

In [26]:
# Build SX order lookup: CustomerID -> set of custpo values
# Filter SX to March only (overlap with SF cases)
sx_march = sx[sx['Enter_Date'].str[:7] == '2026-03'].copy()
print(f"SX orders in March: {sx_march['OrderNumber'].nunique()}")

sx_cust_orders = sx_march.groupby(
    sx_march['CustomerID'].str.replace('.0', '', regex=False)
)['OrderNumber'].nunique().to_dict()

# For pipeline cases with known customer ID, check if that customer has March SX orders
pipeline['has_sx_orders'] = pipeline['acct_clean'].map(
    lambda x: x in sx_cust_orders if pd.notna(x) else False
)

identifiable = pipeline[pipeline['acct_clean'].notna()]
print(f"\nPipeline cases with Account Number: {len(identifiable)}")
print(f"  Customer has SX orders in March: {identifiable['has_sx_orders'].sum()}")
print(f"  Customer has NO SX orders in March: {(~identifiable['has_sx_orders']).sum()}")

SX orders in March: 33514

Pipeline cases with Account Number: 801
  Customer has SX orders in March: 258
  Customer has NO SX orders in March: 543


## Step 12: Export pipeline candidates

Export the case numbers and key fields for the cases going to the pipeline.

In [27]:
export_cols = [
    'Case Number', 'Date/Time Opened', 'inbox', 'Account Number', 'acct_clean',
    'Account Name', 'Contact Phone', 'Contact Email', 'Email From', 'Email Subject',
    'Attachment Count', 'customer_type', 'segment', 'brand'
]

pipeline[export_cols].to_excel('pipeline_candidates.xlsx', index=False)
print(f"Exported {len(pipeline)} pipeline candidates to pipeline_candidates.xlsx")

# Also export excluded with reasons
excluded = sf[sf['excluded']].copy()
excluded[export_cols + ['exclude_reason']].to_excel('excluded_cases.xlsx', index=False)
print(f"Exported {len(excluded)} excluded cases to excluded_cases.xlsx")

Exported 1808 pipeline candidates to pipeline_candidates.xlsx
Exported 1117 excluded cases to excluded_cases.xlsx


## Summary

Run this cell for a clean summary of all results.

In [28]:
print("=" * 60)
print("INSTAWORKER POC SCOPE ESTIMATION SUMMARY")
print("=" * 60)
print(f"\nTotal SF cases:                           {len(sf)}")
print(f"Period: {sf['Date/Time Opened'].str[:10].min()} to {sf['Date/Time Opened'].str[:10].max()}")
print(f"")
print(f"EXCLUDED:                                  {sf['excluded'].sum()}  ({sf['excluded'].mean()*100:.1f}%)")
for reason, count in sf[sf['excluded']]['exclude_reason'].value_counts().items():
    print(f"  {reason}: {count}")
print(f"")
print(f"SEND TO PIPELINE:                          {(~sf['excluded']).sum()}  ({(~sf['excluded']).mean()*100:.1f}%)")
print(f"  Known in-scope segment: {pipeline[pipeline['segment'].isin(includable_segments)].shape[0]}")
print(f"  Pending-review segment: {pipeline[pipeline['segment'].notna() & ~pipeline['segment'].isin(includable_segments)].shape[0]}")
print(f"  Unknown (no acct / not in DB): {pipeline[pipeline['segment'].isna()].shape[0]}")
print(f"")
print(f"Cases per day going to pipeline: {(~sf['excluded']).sum() / sf['Date/Time Opened'].str[:10].nunique():.0f}")
print(f"")
print(f"NOTE: Pending segments (Competitors, Manufacturers,")
print(f"Food Service, Design, Blank, Other, Other Foodservice)")
print(f"are included in pipeline until Joanne/Susan confirm.")

INSTAWORKER POC SCOPE ESTIMATION SUMMARY

Total SF cases:                           2925
Period: 2026-03-06 to 2026-03-27

EXCLUDED:                                  1117  (38.2%)
  Excluded segment: Dealers: 485
  SX-matched: excluded segment: 457
  Not sent to valid inbox: 125
  Brand mismatch: 34
  Excluded segment: Intercompany: 12
  Excluded segment: International: 2
  Excluded segment: Chains: 2

SEND TO PIPELINE:                          1808  (61.8%)
  Known in-scope segment: 166
  Pending-review segment: 93
  Unknown (no acct / not in DB): 1549

Cases per day going to pipeline: 82

NOTE: Pending segments (Competitors, Manufacturers,
Food Service, Design, Blank, Other, Other Foodservice)
are included in pipeline until Joanne/Susan confirm.
